# Lesson 1: One call to the model

**Project:** Agentic Engineering Worklog
**Model:** `qwen3:8b`, running on this PC through Ollama

This lesson is the foundation for everything after it. It sends one message to an AI model from Python and looks closely at what comes back. There is no agent yet. An agent is this same call repeated in a loop, so understanding this one call well makes the later lessons much easier.

This version of the notebook has longer explanations than the first. Every new word is defined the first time it appears, and each step explains its code line by line. The code and its saved output are exactly what ran on this PC.

## Before you start: the big ideas in plain words

**What is an AI model?**
A model is a very large file of numbers that was trained on huge amounts of text. Given some text, it predicts what text should come next, one small piece at a time. That's all it does, but doing it well makes it able to answer questions, follow instructions and write in a particular format. The kind of model used here is called a **large language model**, or **LLM**.

**What is `qwen3:8b`?**
The name of the model used in these lessons. **Qwen3** is a family of free models made by Alibaba. **8b** means about 8 billion numbers inside it (called *parameters*). More parameters usually means a more capable model, but also a bigger file and more memory needed. An 8B model is small enough to run on a home PC.

**What is Ollama?**
A free program that runs models on your own computer. Once it's installed, it runs quietly in the background, like a small web server that only your PC can reach. You ask it to load a model, send it messages, and it sends back the model's replies.

**Local or hosted?**
A *hosted* model (like Claude or ChatGPT) runs on a company's computers, and you pay per use. A *local* model runs on your own PC. It's free to use and private, but slower and less capable. These lessons use a local model on your two GTX 1060 video cards.

**What is the `ollama` Python package?**
A small library of Python code that knows how to talk to the Ollama program. Your notebook uses it so you can write `ollama.chat(...)` instead of building web requests by hand.

**What is a notebook?**
A file (`.ipynb`) made of *cells*. Text cells hold explanations, like this one. Code cells hold Python, and when you run one, its output is saved right underneath it. Running the cells in order matters, because later cells use things that earlier cells created.

## What we are looking for

By the end of this notebook you should be able to say yes to four things:

1. Python can reach the model running on this PC.
2. I know what goes in (a list of messages) and what comes back (a response object).
3. I know the model remembers nothing between calls unless I send the history again.
4. I have one small function, `ask_model`, that does the call for me.

**How to run it in PyCharm:** click a code cell and press **Shift + Enter** to run it and move to the next one. Run the cells in order from top to bottom. If PyCharm asks for an interpreter or kernel, choose this project's `.venv`.

## Step 1: Check that Python can see Ollama

### What we are looking for

Proof that the three pieces are connected: your Python code, the Ollama program, and the model file. If this step fails, nothing else in the notebook can work, so it comes first.

In [26]:
import ollama

MODEL = "qwen3:8b"

downloaded = ollama.list()
for m in downloaded.models:
    size_gb = m.size / 1e9
    print(f"{m.model:<20} {size_gb:.1f} GB")

qwen3:8b             5.2 GB


### The code, line by line

- `import ollama` loads the Ollama Python package so the notebook can use it.
- `MODEL = "qwen3:8b"` stores the model's name in a variable. Writing it once here means every later cell uses the same name, and switching models later means changing one line.
- `downloaded = ollama.list()` asks the Ollama program which models are saved on this PC.
- The `for` loop goes through each model in that list and prints its name and size. `m.size` is in bytes, and dividing by `1e9` (one billion) turns it into gigabytes.

### How to read it

One line per downloaded model. Your run printed:

```
qwen3:8b             5.2 GB
```

So one model is downloaded, and it takes 5.2 GB of disk space.

### What it means

Python reached the Ollama program, and the model is there. If this cell ever fails with a *connection error*, Ollama isn't running: start it from the Windows Start menu and run the cell again.

### What's next and why

Knowing the model exists isn't the same as using it. Step 2 sends it a message.

## Step 2: The first call

### What we are looking for

The smallest complete conversation with the model: one message out, one reply back.

The first run can take 10 to 30 seconds, because the model has to be copied from the disk onto the video cards before it can answer. Later calls are faster, because it stays loaded for a few minutes.

In [27]:
messages = [
    {"role": "user", "content": "In one sentence, what is a work log?"}
]

response = ollama.chat(model=MODEL, messages=messages, think=False)

print(response.message.content)

A work log is a record of an individual's activities, tasks, and time spent on various projects or responsibilities during a specific period.


### The code, line by line

- `messages = [ ... ]` builds a **list** (the square brackets) holding one **dictionary** (the curly braces). A dictionary is a set of labelled values. This one has two labels:
    - `"role": "user"` says who is speaking. `user` means you.
    - `"content": "In one sentence, what is a work log?"` is what you said.
- `ollama.chat(model=MODEL, messages=messages, think=False)` sends the list to the model and waits for the reply. The reply is stored in `response`.
    - `think=False` turns off this model's habit of "thinking out loud" before answering. Thinking can improve hard answers, but it's slower, and these lessons don't need it.
- `print(response.message.content)` prints the text of the reply.

### How to read it

Your run printed:

> A work log is a record of an individual's daily activities, tasks, and time spent on various projects or responsibilities.

If you run the cell again, the wording will probably be a little different. That's normal: models pick their words with some randomness, unless you tell them not to (Step 7 shows how).

### What it means

This is the whole basic pattern, and every later lesson uses it:

1. Build a list of messages.
2. Send it with `ollama.chat`.
3. Read the reply from `response.message.content`.

The reason it's a *list* of messages, and not just one sentence, becomes clear in Steps 5 and 6.

### What's next and why

The reply text is only part of what came back. Step 3 looks at the rest, because it tells you how fast the model is, which matters once an agent makes many calls.

## Step 3: Look at everything that came back

### What we are looking for

What else is inside the response besides the text: who is speaking, how much text went in and out, and how long it took.

In [28]:
print("role:         ", response.message.role)
print("reply:        ", response.message.content)
print()
print("tokens in:    ", response.prompt_eval_count)
print("tokens out:   ", response.eval_count)

seconds_generating = response.eval_duration / 1e9
print("speed:        ", round(response.eval_count / seconds_generating, 1), "tokens per second")
print("total time:   ", round(response.total_duration / 1e9, 1), "seconds")

role:          assistant
reply:         A work log is a record of an individual's activities, tasks, and time spent on various projects or responsibilities during a specific period.

tokens in:     26
tokens out:    28
speed:         17.5 tokens per second
total time:    17.5 seconds


### The code, line by line

- `response.message.role` is who wrote the reply.
- `response.prompt_eval_count` is how many **tokens** the model read. (Tokens are explained below.)
- `response.eval_count` is how many tokens the model wrote.
- `response.eval_duration` is how long the writing took, in **nanoseconds** (billionths of a second). Dividing by `1e9` turns it into seconds.
- Speed is tokens written divided by seconds spent writing.
- `response.total_duration` is the whole time from sending to finishing, including loading the model if it wasn't loaded yet.

### How to read it

Your run printed:

| Line | Your value | What it tells you |
|---|---|---|
| role | `assistant` | The model's role name. `user` is you; `assistant` is the model. |
| tokens in | 26 | The size of what you sent. One short question is 26 tokens. |
| tokens out | 28 | The size of the reply: one sentence. |
| speed | 17.5 tokens per second | How fast the model writes. |
| total time | 17.5 seconds | Most of this was loading the model onto the video cards for the first time. |

### What it means

**What is a token?** Models don't read whole words. They read *tokens*, which are short words or pieces of longer words. "work" might be one token, while "unbelievable" might be three. A rough rule: 100 tokens is about 75 English words.

Your run used more than that rule predicts: an 8-word question became 26 tokens. The extra tokens are hidden markers the model adds around each message, to note where it starts and who it's from.

**Why tokens matter for agents:**

- **Speed.** At 17.5 tokens per second, a 100-token reply takes about 6 seconds. An agent might make 3 or 4 calls to finish one job, so seconds add up.
- **Size limit.** Every model can only read so many tokens at once. That limit caps how much conversation history you can send, which is the topic of lesson 4.
- **Cost.** Hosted models charge per token. Yours is free, but the habit of watching token counts carries over.

**Is 17.5 tokens per second good?** For this PC, yes. 15 or more means the model is running on the video cards. Under about 5 would mean it's running on the main processor, which is much slower. The 17.5 seconds total was mostly a one-time loading cost: writing 28 tokens at 17.5 per second takes under 2 seconds.

**Speed varies a little from run to run.** Earlier runs of this cell gave about 20 to 21 tokens per second. The numbers in this lesson are the ones saved in this run.

### What's next and why

The speed suggests the model is on the video cards. Step 4 checks that directly, instead of guessing.

## Step 4: Check where the model is running

### What we are looking for

How much of the loaded model sits in **video card memory** (also called **VRAM**). Models run far faster on video cards (**GPUs**) than on the main processor (**CPU**), so this tells you whether you're getting full speed.

In [29]:
running = ollama.ps()

if not running.models:
    print("No model is loaded right now. Run the Step 2 cell again, then this one.")

for m in running.models:
    on_gpu = m.size_vram / m.size * 100
    print(f"model:        {m.model}")
    print(f"total size:   {m.size / 1e9:.1f} GB")
    print(f"on the GPUs:  {m.size_vram / 1e9:.1f} GB  ({on_gpu:.0f}%)")

model:        qwen3:8b
total size:   5.9 GB
on the GPUs:  5.9 GB  (100%)


### The code, line by line

- `ollama.ps()` asks Ollama which models are loaded right now. The name comes from the `ps` command, which lists running programs.
- If nothing is loaded, it prints a reminder, because Ollama unloads a model after about 5 idle minutes.
- For each loaded model, `m.size` is the memory it needs and `m.size_vram` is how much of that is in video card memory. Dividing one by the other and multiplying by 100 gives a percentage.

### How to read it

Your run printed:

```
model:        qwen3:8b
total size:   5.9 GB
on the GPUs:  5.9 GB  (100%)
```

- **total size, 5.9 GB:** bigger than the 5.2 GB file from Step 1, because a running model needs extra working space.
- **on the GPUs, 100%:** all of it is in video card memory.

### What it means

100% is the goal: the whole model is on your two GTX 1060 cards, which together have 12 GB of video memory. That's why you got 17.5 tokens per second.

If this ever shows less than 100%, part of the model has spilled into the PC's regular memory and runs on the processor, which is slower. The usual cause is another program using video memory, such as a browser with many tabs or a game.

### What's next and why

You now know the model works and runs at full speed. Step 5 shows the single most important fact about how models work, and the reason agents are built the way they are.

## Step 5: The model has no memory

### What we are looking for

Whether the model remembers something you told it in an earlier call. There are two separate calls: the first tells it a fact, the second asks about that fact.

In [30]:
first = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "I spent this morning on my Cornerpost Watch project."}],
    think=False,
)
print("Call 1 reply:", first.message.content)
print()

second = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "Which project did I work on this morning?"}],
    think=False,
)
print("Call 2 reply:", second.message.content)

Call 1 reply: That sounds like a productive morning! What progress did you make on the Cornerpost Watch project? I’d love to hear more about it — is it a hardware project, a software tool, or something else?

Call 2 reply: I don't have access to your personal work history or calendar. However, you can check:

1. **Your task management tool** (e.g., Trello, Asana, Notion, Jira).
2. **Email or calendar** for any meetings or tasks scheduled this morning.
3. **Project management software** if you use one (e.g., ClickUp, Monday.com).
4. **Notes or to-do lists** you might have created.

Let me know if you'd like help organizing or tracking your tasks!


### The code, line by line

- The first `ollama.chat` call sends one message: "I spent this morning on my Cornerpost Watch project." The reply is stored in `first`.
- The second `ollama.chat` call sends a **new** list with one message: "Which project did I work on this morning?" The reply is stored in `second`.

Notice that the second call's `messages` list doesn't include anything from the first call.

### How to read it

Your run printed (shortened):

> **Call 1 reply:** That sounds like a great project! Could you tell me more about the Cornerpost Watch? ...
>
> **Call 2 reply:** I don't have access to your personal work history or calendar. ...

Call 1 responded to what you said. Call 2 had no idea, even though you told it a moment earlier.

### What it means

**Each call starts from nothing.** The model doesn't store your messages anywhere between calls. The second call only received one sentence, the question, so it had nothing to answer from.

This surprises most people, because chat apps like Claude or ChatGPT seem to remember the conversation. They do it with a trick: on every turn, the app sends the **entire conversation so far** back to the model. The model reads it all fresh each time.

Call 2 is also a good example of an honest answer. It said it didn't know, instead of guessing. Models don't always do that; Step 7 shows one inventing an answer.

### What's next and why

If memory is just "send the conversation again", you can do that yourself. Step 6 does exactly that.

## Step 6: Memory is the message list

### What we are looking for

Whether the same question gets the right answer once the earlier exchange is sent along with it.

In [31]:
history = [
    {"role": "user", "content": "I spent this morning on my Cornerpost Watch project."},
    {"role": "assistant", "content": first.message.content},
    {"role": "user", "content": "Which project did I work on this morning?"},
]

third = ollama.chat(model=MODEL, messages=history, think=False)

print("Reply with history:", third.message.content)
print()
print("Messages sent:", len(history))
print("Tokens in:    ", third.prompt_eval_count)

Reply with history: You mentioned that you spent the morning on your **Cornerpost Watch project**. So, the project you worked on this morning was the **Cornerpost Watch**. 

Would you like to share more about what that project involves?

Messages sent: 3
Tokens in:     88


### The code, line by line

- `history` is a list of **three** messages, in order:
    1. Your first message (`user`): "I spent this morning on my Cornerpost Watch project."
    2. The model's reply to it from Step 5 (`assistant`): `first.message.content` reuses the exact text it said.
    3. Your question (`user`): "Which project did I work on this morning?"
- `ollama.chat(model=MODEL, messages=history, ...)` sends all three at once.
- The last two lines print how many messages were sent and how many tokens the model read.

### How to read it

Your run printed (shortened):

> **Reply with history:** You mentioned that you spent the morning on your **Cornerpost Watch project**. ...
>
> Messages sent: 3
>
> Tokens in: 108

- The answer is now right: **Cornerpost Watch**.
- **Tokens in** went from 26 in Step 3 to **108**, because the model had to read the whole history, including its own fairly long reply from Step 5.

### What it means

**The model's "memory" is whatever is in the `messages` list for that one call, and your code decides what goes in.** Nothing else.

That gives you control, and it has a cost:

- **Control:** you choose what the model sees. You can leave out things that don't matter, or add facts it needs.
- **Cost:** every message you keep is re-read on every call. Here, 3 messages cost 108 tokens. A long conversation could cost thousands, making every reply slower.

This idea is the heart of how an agent works. In later lessons, the list also holds the results of the agent's actions (lesson 2's tool results), and the model reads those results to decide what to do next.

### What's next and why

So far the model has acted as a general chatbot. Step 7 gives it a specific job.

## Step 7: Give the model a job with a system message

### What we are looking for

Whether standing instructions can turn a general chatbot into a specific helper, here a work log assistant that replies in a fixed format.

In [32]:
SYSTEM_PROMPT = (
    "You are a work log assistant. "
    "When the user describes work they did, reply with exactly one line in this form: "
    "LOGGED: <project> | <what was done>. "
    "If the message is not about work, reply with: NOT A LOG ENTRY."
)

tests = [
    "Fixed the alert email bug in Cornerpost Watch.",
    "Spent an hour cleaning the meat price data.",
    "What is the capital of Colorado?",
]

for text in tests:
    reply = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text},
        ],
        think=False,
        options={"temperature": 0},
    )
    print("You said:", text)
    print("Model:   ", reply.message.content)
    print()

You said: Fixed the alert email bug in Cornerpost Watch.
Model:    LOGGED: Cornerpost Watch | Fixed the alert email bug

You said: Spent an hour cleaning the meat price data.
Model:    LOGGED: Meat Price Data | Cleaned the meat price data.

You said: What is the capital of Colorado?
Model:    NOT A LOG ENTRY



### The code, line by line

- `SYSTEM_PROMPT` is the instruction text. It tells the model:
    - its job: "You are a work log assistant."
    - an exact format for work: `LOGGED: <project> | <what was done>`
    - what to do with anything else: reply `NOT A LOG ENTRY`.

  (The text is split over several lines in parentheses only to make it easier to read. Python joins them into one string.)
- `tests` is a list of three sentences to try: two about work, one not.
- The `for` loop sends each sentence with **two** messages:
    1. `{"role": "system", "content": SYSTEM_PROMPT}`: the instructions, always first.
    2. `{"role": "user", "content": text}`: the sentence being tested.
- `options={"temperature": 0}` is new. **Temperature** controls how much randomness the model uses when picking words. At 0, it always picks its most likely choice, so the same input gives the same output. That makes testing fair.

### How to read it

Your run printed:

| You said | Model replied | Right? |
|---|---|---|
| Fixed the alert email bug in Cornerpost Watch. | `LOGGED: Cornerpost Watch \| Fixed the alert email bug` | Yes |
| Spent an hour cleaning the meat price data. | `LOGGED: Meat Price Data \| Cleaned the meat price data.` | Format right, project made up |
| What is the capital of Colorado? | `NOT A LOG ENTRY` | Yes |

### What it means

There are now three **roles** a message can have:

- `system`: standing instructions, sent first.
- `user`: you.
- `assistant`: the model.

The system message worked: the model followed the format all three times. But look closely at two things:

1. **It invented a project name.** "Meat Price Data" isn't one of your projects. The model had no list of your real projects, so it made one up from the sentence. Models fill gaps with plausible guesses unless you give them the facts. Lesson 2 fixes this by giving it your real project names.
2. **Nothing was saved.** The model wrote text that *looks* like a log entry, but no record exists anywhere. A model can only produce text. To actually save something, it needs a way to ask your Python code to do it. That's called a **tool**, and it's lesson 2.

### What's next and why

Every lesson from here calls the model the same way, so Step 8 packages the call into one reusable function.

## Step 8: Wrap the call in one function

### What we are looking for

A short, reusable way to call the model, and a tiny chat program built from it that shows memory working over several turns.

In [33]:
def ask_model(messages, model=MODEL):
    """Send a list of messages to the local model and return its reply text."""
    response = ollama.chat(
        model=model,
        messages=messages,
        think=False,
        options={"temperature": 0},
    )
    return response.message.content


conversation = [{"role": "system", "content": "You are a helpful assistant. Answer in one short sentence."}]

for question in ["What is a token?", "Give me one example."]:
    conversation.append({"role": "user", "content": question})
    answer = ask_model(conversation)
    conversation.append({"role": "assistant", "content": answer})
    print("You:  ", question)
    print("Model:", answer)
    print()

print("Messages in the conversation now:", len(conversation))

You:   What is a token?
Model: A token is a digital representation of value used in blockchain and cryptocurrency systems to represent assets, rights, or utilities.

You:   Give me one example.
Model: Bitcoin is an example of a token, specifically a cryptocurrency token used as a medium of exchange.

Messages in the conversation now: 5


### The code, line by line

- `def ask_model(messages, model=MODEL):` defines a **function**: a named, reusable block of code. You give it a list of messages; it gives back the reply text.
    - The text in triple quotes is a **docstring**: a short note saying what the function does.
    - Inside, it makes the same `ollama.chat` call as before, with thinking off and temperature 0.
    - `return response.message.content` hands back just the reply text.
- `conversation` starts as a list with one `system` message: "Answer in one short sentence."
- The `for` loop asks two questions. For each one it:
    1. adds your question to the list (`append` adds an item to the end of a list),
    2. calls `ask_model` with the **whole** list,
    3. adds the model's answer to the list,
    4. prints both.

### How to read it

Your run printed:

> **You:** What is a token?
>
> **Model:** A token is a digital representation of value used in blockchain and cryptocurrency systems ...
>
> **You:** Give me one example.
>
> **Model:** Bitcoin is an example of a token ...
>
> Messages in the conversation now: 5

Two things to notice:

- **Memory worked.** "Give me one example" only makes sense next to the first question, and the model connected them. The list ended with 5 messages: 1 system, 2 from you, 2 from the model.
- **It guessed the wrong meaning of "token."** You meant AI tokens (from Step 3), but the model answered about cryptocurrency. Nothing in *this* conversation mentioned AI. The earlier steps were separate calls it never saw. Its answer was reasonable, just not what you meant.

### What it means

That small `for` loop is a complete chat program: add your message, call the model, add its reply, repeat. The growing list is the conversation's memory.

The token mix-up is the same lesson as Step 5, seen from another angle: **the model only knows what's in the list you send.** Context you know, but didn't send, doesn't exist for it.

An agent's loop has this same shape. The difference, starting in lesson 3, is that some turns aren't you talking. They're the model asking for a tool, and your code adding the tool's result to the list.

## Try it yourself

Change one thing at a time and rerun the cell:

1. In Step 7, add a fourth test sentence about one of your own projects. Does it get logged with the right project name?
2. In Step 7, change the format to include hours: `LOGGED: <project> | <what was done> | <hours>`. What does the model do when you don't mention hours? (This shows a real weakness: when information is missing, a model may invent it.)
3. In Step 8, change the system message to "You are a helpful assistant. We are talking about AI language models. Answer in one short sentence." Does "What is a token?" get the AI meaning now?

## What it all means

- A model call is simple: a list of messages in, one message out.
- The model keeps nothing between calls. **The message list is its only memory, and your code controls that list.**
- A `system` message gives the model a job and a format.
- Models fill gaps with guesses, like the made-up "Meat Price Data" and the cryptocurrency "token", unless you give them the facts.
- A model can only produce text. It can't save, look up, or change anything on its own.

## Words used in this lesson

| Word | Meaning |
|---|---|
| Model / LLM | A large file of numbers trained on text, which predicts what text comes next. |
| Ollama | A free program that runs models on your own PC. |
| Token | The piece of text a model reads and writes: a short word or part of a word. |
| Message | One item in the list sent to the model, with a `role` and `content`. |
| Role | Who a message is from: `system` (instructions), `user` (you) or `assistant` (the model). |
| System prompt | The standing instructions, sent as the first message. |
| Temperature | How much randomness the model uses when choosing words. 0 means none. |
| GPU / VRAM | The video card and its memory. Models run much faster there than on the main processor. |
| Function | A named, reusable block of code, like `ask_model`. |

## What's next and why

**Lesson 2: One tool.** You'll write a Python function, `add_entry`, that saves a work log entry to a file, and describe it to the model. The model can then *ask* for that function to be run, with the project and description filled in, and your code does the saving.

That is the step from "text that looks like a record" to "a record that exists". It's the first half of what makes an agent. Lesson 3 adds the loop, which is the second half.